In [ ]:
#| default_exp handlers.pipeline.intake

# Intake

Boundary ingestion planning and execution for default CSV/TSV sources, with an explicit Early Excel Intake Gate and custom-loader delegation.

In [ ]:
#| export
from __future__ import annotations
import io
from pathlib import Path
from typing import Any, Optional
import pandas as pd
import requests
from pydantic import BaseModel, Field
from marisco.handlers.pipeline.contracts import HandlerConfig, PluginSpec
from marisco.handlers.pipeline.gates import _custom_loader_skeleton


## Intake Plan

In [ ]:
#| export
class IntakePlan(BaseModel):
    "Describe the boundary ingestion route chosen for a handler config."
    kind: str
    fmt: str
    grp: str = "SEAWATER"
    sep: str = ","


class LoaderExecutionContext(BaseModel):
    "Structured context attached to failures at the boundary-loader seam."
    title: str
    module_name: str
    spec: dict[str, Any] = Field(default_factory=dict)
    args: dict[str, Any] = Field(default_factory=dict)

    @classmethod
    def from_config(cls, cfg: HandlerConfig) -> "LoaderExecutionContext":
        loader = cfg.loader
        return cls(
            title=cfg.title,
            module_name=cfg.module_name,
            spec=loader.model_dump(mode="json", by_alias=True, exclude_none=True) if loader else {},
            args=loader.args if loader else {},
        )


class LoaderExecutionError(RuntimeError):
    "Boundary-loader failure enriched with a Pydantic-validated context payload."
    def __init__(self, context: LoaderExecutionContext):
        self.context = context
        super().__init__(context.model_dump_json(indent=2))


def _unsupported_default_loader_message(cfg: HandlerConfig) -> str:
    fmt = (cfg.fmt or "csv").lower()
    sections = [
        f"Intake Gate failed: format '{fmt}' cannot be loaded by the default CSV reader.",
        "The default ingestion path only supports text-delimited CSV/TSV files.",
        "Excel workbooks require a Custom Boundary Loader before the declarative pipeline can begin.",
        _custom_loader_skeleton(cfg, findings=[{'grp': 'SEAWATER'}]),
    ]
    return "\n\n".join(part for part in sections if part)


def resolve_intake_plan(cfg: HandlerConfig, grp: str = "SEAWATER") -> IntakePlan:
    fmt = (cfg.fmt or "csv").lower()
    if fmt in {"xlsx", "xls", "excel"}:
        return IntakePlan(kind="unsupported_excel", fmt=fmt, grp=grp)
    sep = "\t" if fmt == "tsv" else ","
    return IntakePlan(kind="delimited", fmt=fmt, grp=grp, sep=sep)


def execute_intake_plan(cfg: HandlerConfig, plan: IntakePlan) -> dict[str, pd.DataFrame]:
    if plan.kind == "unsupported_excel":
        msg = _unsupported_default_loader_message(cfg)
        print(f"\n{msg}\n")
        raise ValueError(msg)
    r = requests.get(cfg.url, timeout=60)
    r.raise_for_status()
    return {plan.grp: pd.read_csv(io.BytesIO(r.content), sep=plan.sep)}


def load_data(cfg: HandlerConfig, grp: str = "SEAWATER") -> dict[str, pd.DataFrame]:
    "Fetch raw provider data through the default boundary-ingestion route."
    plan = resolve_intake_plan(cfg, grp=grp)
    return execute_intake_plan(cfg, plan)


def resolve_loader_fn(spec: PluginSpec, yaml_dir: Path = None):
    "Resolve a custom boundary-loader function from a PluginSpec."
    return spec.resolve_fn(yaml_dir)


def call_loader(cfg: HandlerConfig, yaml_dir: Optional[Path] = None):
    "Execute the configured loader and retain its root cause with boundary context."
    context = LoaderExecutionContext.from_config(cfg)
    try:
        if not cfg.loader:
            return load_data(cfg)
        loader_fn = resolve_loader_fn(cfg.loader, yaml_dir=yaml_dir)
        return loader_fn(cfg, **cfg.loader.args)
    except Exception as exc:
        raise LoaderExecutionError(context) from exc


In [ ]:
cfg = HandlerConfig(
    module_name="self_contained_fixture",
    title="Self-contained pipeline fixture",
    url="memory://pipeline-fixture",
    fname_out="pipeline-fixture.nc",
    columns={"provider_lat": "LAT", "provider_lon": "LON"},
    col_date="provider_time",
    melt_spec=[{
        "nuclide": "Cs-137",
        "val": "provider_value",
        "unc": "provider_unc",
        "unit": "Bq/m3",
        "lab": "Fixture lab",
    }],
)
plan = resolve_intake_plan(cfg)
print(plan.kind)
print(plan.fmt)


In [ ]:
#| test
cfg = HandlerConfig(
    module_name="broken_loader",
    title="Broken loader fixture",
    url="https://example.invalid/source.csv",
    fname_out="broken.nc",
    loader={"path": "marisco.handlers.pipeline.intake.load_data", "args": {"sheet": "raw"}},
)


def _broken_loader(cfg, **kwargs):
    raise FileNotFoundError("fixture source is absent")


_original_resolve_loader_fn = resolve_loader_fn
resolve_loader_fn = lambda *args, **kwargs: _broken_loader
try:
    call_loader(cfg)
except LoaderExecutionError as exc:
    assert isinstance(exc.context, LoaderExecutionContext)
    assert exc.context.title == "Broken loader fixture"
    assert exc.context.module_name == "broken_loader"
    assert exc.context.args == {"sheet": "raw"}
    assert exc.context.spec["path"] == "marisco.handlers.pipeline.intake.load_data"
    assert isinstance(exc.__cause__, FileNotFoundError)
else:
    raise AssertionError("Loader failures must preserve the root cause through chaining.")
finally:
    resolve_loader_fn = _original_resolve_loader_fn
